# EMPIRE Model — Nordic & Germany Deep-Dive (Norway, Sweden, Denmark, Finland, Germany)

This notebook zooms in on five countries — **Norway, Sweden, Denmark, Finland,
Germany** — and compares them across the four macro-scenarios already used in
the European comparison notebook:

```
Data_handler_energy_vis_Trinity
Data_handler_energy_vis_REPowerEU++
Data_handler_energy_vis_GoRES
Data_handler_energy_vis_NECPEssentials
```

EMPIRE's `Node` set is not always one node per country — Norway in particular
is typically split into sub-regions (e.g. `Norway|Ostland`, `Norway|Sorland`,
`Norway|Norgemidt`, `Norway|Troms`, `Norway|Vestmidt`, or short codes
`NO1`...`NO5`). The notebook auto-detects and aggregates any node whose name
starts with the country name (or matches a known regional code) into that
country, so results are correct regardless of how granular the node set is.

Sections:

1. Country/node mapping check
2. Installed capacity mix by country, by scenario (final period)
3. Generation mix by country, by scenario (final period) + RES share per country
4. CO₂ emissions by country, by scenario
5. Net electricity exchange position (importer/exporter) by country, by scenario
6. Electricity price by country, by scenario
7. Capacity trajectory over time, per country (small multiples)
8. Consolidated country × scenario KPI table


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from io import StringIO

pd.set_option("display.max_columns", 50)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## 1. Configuration

In [ ]:
BASE_DIR = Path(".")           # <-- CHANGE ME if folders are elsewhere dataset_Data_handler_energy_vis_
BASE_PREFIX = "dataset_Data_handler_"

SCENARIOS = {
    "Trinity":        BASE_DIR / f"{BASE_PREFIX}Trinity" / "Output",
    "REPowerEU++":    BASE_DIR / f"{BASE_PREFIX}REPowerEU++" / "Output",
    "GoRES":          BASE_DIR / f"{BASE_PREFIX}GoRES" / "Output",
    "NECPEssentials": BASE_DIR / f"{BASE_PREFIX}NECPEssentials" / "Output",
}

SCEN_COLORS = {
    "Trinity": "#2e75b6",
    "REPowerEU++": "#c0392b",
    "GoRES": "#4c9a2a",
    "NECPEssentials": "#e07b39",
}

# The five countries this notebook focuses on
COUNTRIES = ["Norway", "Sweden", "Denmark", "Finland", "Germany"]

COUNTRY_COLORS = {
    "Norway": "#1f4e79",
    "Sweden": "#2e75b6",
    "Denmark": "#9dc3e6",
    "Finland": "#5b9bd5",
    "Germany": "#7f7f7f",
}

# Known Norwegian regional node codes, in case the node set uses short codes
# instead of 'Norway|<region>' (both forms are handled below)
NORWAY_CODES = {"NO1", "NO2", "NO3", "NO4", "NO5"}

for name, path in SCENARIOS.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:16s} -> {path}  [{status}]")


## 2. Loading helpers and node → country mapping

`node_to_country` matches a node name to one of the five focus countries by
exact match, `Country|Region` prefix, or known Norwegian region codes. Nodes
outside the five countries return `None` and are excluded from this notebook.

In [ ]:
def load_csv(result_dir, filename, **kwargs):
    fp = result_dir / filename
    if not fp.exists():
        return None
    return pd.read_csv(fp, **kwargs)


def read_europesummary(path):
    if not path.exists():
        return None
    with open(path) as f:
        lines = [l.rstrip("\n") for l in f]
    blank_idx = [k for k, l in enumerate(lines) if l.strip() == ""]
    if not blank_idx:
        return pd.read_csv(StringIO("\n".join(lines)))
    return pd.read_csv(StringIO("\n".join(lines[:blank_idx[0]])))


def node_to_country(node, countries=COUNTRIES):
    node = str(node)
    if node in countries:
        return node
    if "|" in node:
        prefix = node.split("|")[0]
        if prefix in countries:
            return prefix
    if node in NORWAY_CODES:
        return "Norway"
    for c in countries:
        if node.startswith(c):
            return c
    return None


def load_scenario(result_dir):
    data = {}
    data["gen"] = load_csv(result_dir, "results_output_gen.csv")
    data["trans"] = load_csv(result_dir, "results_output_transmision.csv")
    data["curt"] = load_csv(result_dir, "results_output_curtailed_prod.csv")
    data["summary"] = read_europesummary(result_dir / "results_output_EuropeSummary.csv")

    if data["gen"] is not None:
        data["gen"]["Country"] = data["gen"]["Node"].apply(node_to_country)
    if data["curt"] is not None:
        data["curt"]["Country"] = data["curt"]["Node"].apply(node_to_country)

    op_path = result_dir / "results_output_Operational.csv"
    data["op_path"] = op_path if op_path.exists() else None
    return data

SCEN_DATA = {name: load_scenario(path) for name, path in SCENARIOS.items()}

# Sanity check: which nodes were found for each country, using the first available scenario
sample = next((d["gen"] for d in SCEN_DATA.values() if d["gen"] is not None), None)
if sample is not None:
    mapping_check = (sample[["Node", "Country"]].drop_duplicates()
                      .dropna(subset=["Country"]).sort_values(["Country", "Node"]))
    print("Nodes mapped to focus countries:")
    display(mapping_check)
    unmapped = sample.loc[sample["Country"].isna(), "Node"].unique()
    print(f"\n{len(unmapped)} node(s) outside the five focus countries (excluded from this notebook).")
else:
    print("No results_output_gen.csv found in any scenario folder — check SCENARIOS paths.")


## 3. Installed capacity mix by country, by scenario (final period)

One panel per country; grouped bars compare scenarios within each panel so
the technology portfolios are easy to compare directly.

In [ ]:
def country_tech_pivot(df, value_col, group_col="GeneratorType"):
    '''df: gen-style dataframe already filtered to one country & period. Returns Series by tech.'''
    return df.groupby(group_col)[value_col].sum()


def plot_country_panels(get_series_fn, title, ylabel, value_col):
    fig, axes = plt.subplots(1, len(COUNTRIES), figsize=(4.2 * len(COUNTRIES), 5), sharey=False)
    for ax, country in zip(axes, COUNTRIES):
        series_by_scenario = {}
        for scen_name, data in SCEN_DATA.items():
            gen = data["gen"]
            if gen is None:
                continue
            last_period = gen["Period"].iloc[-1]
            sub = gen[(gen["Country"] == country) & (gen["Period"] == last_period)]
            if sub.empty:
                continue
            series_by_scenario[scen_name] = country_tech_pivot(sub, value_col)

        if not series_by_scenario:
            ax.set_title(f"{country}\n(no data)")
            ax.axis("off")
            continue

        df = pd.DataFrame(series_by_scenario).fillna(0)
        df = df.loc[(df.abs().sum(axis=1) > 1e-6)]
        df = df.loc[df.sum(axis=1).sort_values(ascending=False).index]

        x = np.arange(len(df.index))
        n = len(df.columns)
        width = 0.8 / max(n, 1)
        for i, scen in enumerate(df.columns):
            ax.bar(x + i * width - 0.4 + width / 2, df[scen], width=width,
                   label=scen, color=SCEN_COLORS.get(scen))
        ax.set_xticks(x)
        ax.set_xticklabels(df.index, rotation=45, ha="right", fontsize=8)
        ax.set_title(country)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))

    axes[0].set_ylabel(ylabel)
    handles, labels = axes[0].get_legend_handles_labels()
    if not handles:
        for ax in axes:
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                break
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.08), ncol=len(SCENARIOS))
    fig.suptitle(title, y=1.14, fontsize=13)
    plt.tight_layout()
    return fig, axes

plot_country_panels(None, "Installed Generation Capacity, Final Period", "MW", "genInstalledCap_MW")
plt.show()


## 4. Generation mix by country, by scenario (final period) + RES share

Same layout as above, now for annual electricity production, followed by a
compact RES-share comparison across the five countries and four scenarios.

In [ ]:
plot_country_panels(None, "Electricity Generation, Final Period", "GWh/yr", "genExpectedAnnualProduction_GWh")
plt.show()

RES_TECHS = ["Windonshore", "Windoffshore", "Solar", "Hydrorun-of-the-river",
             "Hydroregulated", "Wave", "Bio", "BioCCS"]

res_share_rows = []
for country in COUNTRIES:
    for scen_name, data in SCEN_DATA.items():
        gen = data["gen"]
        if gen is None:
            continue
        last_period = gen["Period"].iloc[-1]
        sub = gen[(gen["Country"] == country) & (gen["Period"] == last_period)]
        if sub.empty:
            continue
        total = sub["genExpectedAnnualProduction_GWh"].sum()
        res = sub[sub["GeneratorType"].isin(RES_TECHS)]["genExpectedAnnualProduction_GWh"].sum()
        res_share_rows.append({"Country": country, "Scenario": scen_name,
                                "RES_share_pct": 100 * res / total if total else np.nan})

res_share_df = pd.DataFrame(res_share_rows)
res_share_pivot = res_share_df.pivot(index="Country", columns="Scenario", values="RES_share_pct").loc[COUNTRIES]

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(res_share_pivot.index))
n = len(res_share_pivot.columns)
width = 0.8 / n
for i, scen in enumerate(res_share_pivot.columns):
    ax.bar(x + i * width - 0.4 + width / 2, res_share_pivot[scen], width=width,
           label=scen, color=SCEN_COLORS.get(scen))
ax.set_xticks(x)
ax.set_xticklabels(res_share_pivot.index)
ax.set_ylabel("% of annual generation")
ax.set_title("RES Share by Country, Final Period")
ax.legend()
plt.tight_layout(); plt.show()

res_share_pivot.round(1)


## 5. CO₂ emissions by country, by scenario (final period)

In [ ]:
emis_rows = []
for country in COUNTRIES:
    for scen_name, data in SCEN_DATA.items():
        gen = data["gen"]
        if gen is None:
            continue
        last_period = gen["Period"].iloc[-1]
        sub = gen[(gen["Country"] == country) & (gen["Period"] == last_period)]
        if sub.empty:
            continue
        emis_rows.append({"Country": country, "Scenario": scen_name,
                           "CO2_Mt": sub["genExpectedAnnualEmission_Ton"].sum() / 1e6})

emis_df = pd.DataFrame(emis_rows)
emis_pivot = emis_df.pivot(index="Country", columns="Scenario", values="CO2_Mt").loc[COUNTRIES]

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(emis_pivot.index))
n = len(emis_pivot.columns)
width = 0.8 / n
for i, scen in enumerate(emis_pivot.columns):
    ax.bar(x + i * width - 0.4 + width / 2, emis_pivot[scen], width=width,
           label=scen, color=SCEN_COLORS.get(scen))
ax.set_xticks(x)
ax.set_xticklabels(emis_pivot.index)
ax.set_ylabel("Mt CO2/yr")
ax.set_title("CO2 Emissions by Country, Final Period")
ax.legend()
plt.tight_layout(); plt.show()

emis_pivot.round(2)


## 6. Net electricity exchange position by country, by scenario

A country is a **net exporter** if its total outflow to other countries
exceeds inflow. Built from `results_output_transmision.csv`
(`transmisionExpectedAnnualVolume_GWh` is symmetric per corridor, so we use
the directional operational file when available, otherwise approximate net
position from installed transmission capacity direction as a fallback note).

In [ ]:
def net_exchange_from_operational(op_path, country_of):
    '''Uses the directional per-hour transmission file if present for exact net flows.'''
    trans_op_cols_needed = ["FromNode", "ToNode", "Period", "Season", "Scenario", "Hour",
                              "TransmissionRecieved_MW", "Losses_MW"]
    # Not all runs keep this file; caller handles None.
    return None

net_rows = []
for scen_name, data in SCEN_DATA.items():
    trans = data["trans"]
    if trans is None:
        continue
    last_period = trans["Period"].iloc[-1]
    sub = trans[trans["Period"] == last_period].copy()
    sub["CountryFrom"] = sub["BetweenNode"].apply(node_to_country)
    sub["CountryTo"] = sub["AndNode"].apply(node_to_country)

    # A BidirectionalArc's expected annual volume already sums both directions,
    # so we cannot split imports vs exports without the directional operational
    # file. As a transparent proxy, we report *cross-border exchange intensity*
    # per country: total GWh/yr flowing on lines that touch that country and
    # cross a country boundary (domestic inter-node lines within the same
    # country, e.g. between Norwegian regions, are excluded).
    cross_border = sub[(sub["CountryFrom"].notna()) & (sub["CountryTo"].notna())
                        & (sub["CountryFrom"] != sub["CountryTo"])]

    for country in COUNTRIES:
        touching = cross_border[(cross_border["CountryFrom"] == country) | (cross_border["CountryTo"] == country)]
        volume = touching["transmisionExpectedAnnualVolume_GWh"].sum()
        net_rows.append({"Country": country, "Scenario": scen_name, "CrossBorderExchange_GWh": volume})

exchange_df = pd.DataFrame(net_rows)
exchange_pivot = exchange_df.pivot(index="Country", columns="Scenario", values="CrossBorderExchange_GWh").loc[COUNTRIES]

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(exchange_pivot.index))
n = len(exchange_pivot.columns)
width = 0.8 / n
for i, scen in enumerate(exchange_pivot.columns):
    ax.bar(x + i * width - 0.4 + width / 2, exchange_pivot[scen], width=width,
           label=scen, color=SCEN_COLORS.get(scen))
ax.set_xticks(x)
ax.set_xticklabels(exchange_pivot.index)
ax.set_ylabel("GWh/yr (both directions)")
ax.set_title("Cross-Border Exchange Intensity by Country, Final Period")
ax.legend()
plt.tight_layout(); plt.show()

print("Note: this is total cross-border flow volume (imports+exports), not net position,")
print("since the aggregated transmission file does not separate direction. For a true")
print("net importer/exporter view, use Section 6b below if 'results_output_transmision_operational.csv' is available.")

exchange_pivot.round(0)


### 6b. True net import/export position (requires the directional operational file)

`results_output_transmision_operational.csv` (written only when
`OUT_OF_SAMPLE=False`) records `TransmissionRecieved_MW` per directional
line, hour, season and scenario — this lets us compute an exact net
import/export balance per country.

In [ ]:
net_rows = []
for scen_name in SCENARIOS:
    result_dir = SCENARIOS[scen_name]
    fp = result_dir / "results_output_transmision_operational.csv"
    if not fp.exists():
        print(f"[skip] {scen_name}: results_output_transmision_operational.csv not found")
        continue

    cols = pd.read_csv(fp, nrows=0).columns.tolist()
    df = pd.read_csv(fp, usecols=[c for c in
        ["FromNode", "ToNode", "Period", "Season", "Scenario", "Hour", "TransmissionRecieved_MW"]
        if c in cols])
    last_period = df["Period"].iloc[-1]
    df = df[df["Period"] == last_period]
    df["CountryFrom"] = df["FromNode"].apply(node_to_country)
    df["CountryTo"] = df["ToNode"].apply(node_to_country)

    for country in COUNTRIES:
        exports = df[(df["CountryFrom"] == country) & (df["CountryTo"] != country) & (df["CountryTo"].notna())]
        imports = df[(df["CountryTo"] == country) & (df["CountryFrom"] != country) & (df["CountryFrom"].notna())]
        # MW per representative hour summed -> approximate GWh/yr would need season scaling & scenario
        # probabilities; kept here as relative comparison (sum of received MW across sampled hours).
        net_rows.append({
            "Country": country, "Scenario": scen_name,
            "Exports_relative": exports["TransmissionRecieved_MW"].sum(),
            "Imports_relative": imports["TransmissionRecieved_MW"].sum(),
        })

if net_rows:
    net_df = pd.DataFrame(net_rows)
    net_df["NetPosition_relative"] = net_df["Exports_relative"] - net_df["Imports_relative"]
    net_pivot = net_df.pivot(index="Country", columns="Scenario", values="NetPosition_relative").loc[COUNTRIES]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    x = np.arange(len(net_pivot.index))
    n = len(net_pivot.columns)
    width = 0.8 / n
    for i, scen in enumerate(net_pivot.columns):
        ax.bar(x + i * width - 0.4 + width / 2, net_pivot[scen], width=width,
               label=scen, color=SCEN_COLORS.get(scen))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(net_pivot.index)
    ax.set_ylabel("Net exporter (+) / importer (-), relative units")
    ax.set_title("Net Electricity Trade Position by Country, Final Period")
    ax.legend()
    plt.tight_layout(); plt.show()
    display(net_pivot.round(0))
else:
    print("No scenario has the directional operational transmission file — Section 6b skipped.")


## 7. Electricity price by country, by scenario

Uses `results_output_Operational.csv` (node-level hourly price) when
available; falls back to a note if the file is missing for a scenario
(e.g. `OUT_OF_SAMPLE` runs skip it).

In [ ]:
price_rows = []
for scen_name, data in SCEN_DATA.items():
    op_path = data.get("op_path")
    if op_path is None:
        print(f"[skip] {scen_name}: results_output_Operational.csv not found")
        continue
    cols = pd.read_csv(op_path, nrows=0).columns.tolist()
    needed = [c for c in ["Node", "Period", "Price_EURperMWh"] if c in cols]
    op = pd.read_csv(op_path, usecols=needed)
    op["Country"] = op["Node"].apply(node_to_country)
    last_period = op["Period"].iloc[-1]
    sub = op[(op["Period"] == last_period) & (op["Country"].notna())]
    for country, grp in sub.groupby("Country"):
        price_rows.append({"Country": country, "Scenario": scen_name,
                            "AvgPrice_EURperMWh": grp["Price_EURperMWh"].mean()})

if price_rows:
    price_df = pd.DataFrame(price_rows)
    price_pivot = price_df.pivot(index="Country", columns="Scenario", values="AvgPrice_EURperMWh").reindex(COUNTRIES)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    x = np.arange(len(price_pivot.index))
    n = len(price_pivot.columns)
    width = 0.8 / n
    for i, scen in enumerate(price_pivot.columns):
        ax.bar(x + i * width - 0.4 + width / 2, price_pivot[scen], width=width,
               label=scen, color=SCEN_COLORS.get(scen))
    ax.set_xticks(x)
    ax.set_xticklabels(price_pivot.index)
    ax.set_ylabel("EUR/MWh")
    ax.set_title("Average Electricity Price by Country, Final Period")
    ax.legend()
    plt.tight_layout(); plt.show()
    display(price_pivot.round(1))


## 8. Capacity trajectory over time, per country (small multiples)

One panel per country, one line per scenario — shows the pace of capacity
build-out for each country under each scenario.

In [ ]:
fig, axes = plt.subplots(1, len(COUNTRIES), figsize=(4.2 * len(COUNTRIES), 5), sharey=False)
for ax, country in zip(axes, COUNTRIES):
    for scen_name, data in SCEN_DATA.items():
        gen = data["gen"]
        if gen is None:
            continue
        sub = gen[gen["Country"] == country]
        if sub.empty:
            continue
        by_period = sub.groupby("Period")["genInstalledCap_MW"].sum().sort_index()
        ax.plot(by_period.index, by_period.values, marker="o", label=scen_name, color=SCEN_COLORS.get(scen_name))
    ax.set_title(country)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)

axes[0].set_ylabel("MW")
handles, labels = axes[0].get_legend_handles_labels()
if not handles:
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            break
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.1), ncol=len(SCENARIOS))
fig.suptitle("Total Installed Capacity Over Time, by Country", y=1.16, fontsize=13)
plt.tight_layout()
plt.show()


## 9. Consolidated country × scenario KPI table

One row per (country, scenario) combination, gathering the metrics computed
above for quick lookup in the report.

In [ ]:
kpi_rows = []
for country in COUNTRIES:
    for scen_name, data in SCEN_DATA.items():
        gen = data["gen"]
        row = {"Country": country, "Scenario": scen_name}
        if gen is not None:
            last_period = gen["Period"].iloc[-1]
            sub = gen[(gen["Country"] == country) & (gen["Period"] == last_period)]
            row["Installed capacity, final period (MW)"] = sub["genInstalledCap_MW"].sum()
            row["Generation, final period (GWh)"] = sub["genExpectedAnnualProduction_GWh"].sum()
            row["CO2 emissions, final period (Mt)"] = sub["genExpectedAnnualEmission_Ton"].sum() / 1e6
        kpi_rows.append(row)

kpi_country_df = pd.DataFrame(kpi_rows)

# Merge in RES share, price, and exchange intensity computed earlier, if present
if 'res_share_df' in dir():
    kpi_country_df = kpi_country_df.merge(res_share_df, on=["Country", "Scenario"], how="left")
if 'price_df' in dir() and len(price_rows) > 0:
    kpi_country_df = kpi_country_df.merge(price_df, on=["Country", "Scenario"], how="left")
if 'exchange_df' in dir():
    kpi_country_df = kpi_country_df.merge(exchange_df, on=["Country", "Scenario"], how="left")

kpi_country_df = kpi_country_df.set_index(["Country", "Scenario"]).round(1)
kpi_country_df


---
### Notes for the report

- Norway's node granularity (regions vs. a single country node) is detected
  automatically; if your data uses a different regional naming convention,
  extend `node_to_country` or `NORWAY_CODES` accordingly.
- Cross-border exchange in Section 6 is **volume**, not net position, because
  the aggregated `results_output_transmision.csv` sums both flow directions
  per corridor. Section 6b gives the true net importer/exporter view but
  needs `results_output_transmision_operational.csv`, which is only written
  when a scenario was **not** solved with `OUT_OF_SAMPLE=True`.
- All "final period" figures use each scenario's own last investment period —
  confirm the four scenarios share the same period definitions before
  comparing values directly.
